In [ ]:
# ШАГ 1: Установка и подключение Drive

!pip install vk_api google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

from google.colab import drive
drive.mount('/content/drive')

import os
BOT_FOLDER = '/content/drive/MyDrive/StudyBot'
if not os.path.exists(BOT_FOLDER):
    os.makedirs(BOT_FOLDER)

In [ ]:
# ШАГ 2: Минимальная БД

import sqlite3
from datetime import datetime

class Database:
    def __init__(self):
        self.conn = sqlite3.connect(os.path.join(BOT_FOLDER, 'bot.db'))
        self.cursor = self.conn.cursor()
        self._init_db()

    def _init_db(self):
        # Только пользователи и их состояние
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS users (
                user_id INTEGER PRIMARY KEY,
                state TEXT DEFAULT 'menu'
            )
        ''')
        self.conn.commit()

    def get_user(self, user_id):
        self.cursor.execute('SELECT state FROM users WHERE user_id = ?', (user_id,))
        row = self.cursor.fetchone()
        if not row:
            self.cursor.execute('INSERT INTO users (user_id) VALUES (?)', (user_id,))
            self.conn.commit()
            return 'menu'
        return row[0]

    def set_state(self, user_id, state):
        self.cursor.execute('UPDATE users SET state = ? WHERE user_id = ?', (state, user_id))
        self.conn.commit()

db = Database()

In [ ]:
# ШАГ 3: Google Calendar подключение

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import pickle

class GoogleCalendar:
    def __init__(self):
        self.service = None
        self.creds_file = os.path.join(BOT_FOLDER, 'credentials.json')
        self.token_file = os.path.join(BOT_FOLDER, 'token.pickle')
        self.SCOPES = ['https://www.googleapis.com/auth/calendar']

    def authenticate(self):
        creds = None

        if os.path.exists(self.token_file):
            with open(self.token_file, 'rb') as token:
                creds = pickle.load(token)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(self.creds_file, self.SCOPES)
                creds = flow.run_local_server(port=0)

            with open(self.token_file, 'wb') as token:
                pickle.dump(creds, token)

        self.service = build('calendar', 'v3', credentials=creds)
        return self.service is not None

    def add_event(self, summary, description, date, time):
        if not self.service:
            return False

        from datetime import datetime as dt
        start_datetime = dt.strptime(f"{date} {time}", "%Y-%m-%d %H:%M")
        end_datetime = dt.strptime(f"{date} {time}", "%Y-%m-%d %H:%M")

        event = {
            'summary': summary,
            'description': description,
            'start': {'dateTime': start_datetime.isoformat(), 'timeZone': 'Europe/Moscow'},
            'end': {'dateTime': end_datetime.isoformat(), 'timeZone': 'Europe/Moscow'},
        }

        self.service.events().insert(calendarId='primary', body=event).execute()
        return True

    def get_events(self, days_ahead=7):
        if not self.service:
            return []

        from datetime import datetime, timedelta
        now = datetime.utcnow().isoformat() + 'Z'
        later = (datetime.utcnow() + timedelta(days=days_ahead)).isoformat() + 'Z'

        events = self.service.events().list(
            calendarId='primary', timeMin=now, timeMax=later,
            singleEvents=True, orderBy='startTime'
        ).execute()

        return events.get('items', [])

calendar = GoogleCalendar()

In [ ]:
# ШАГ 4: VK бот с минимальным функционалом

import vk_api
from vk_api.longpoll import VkLongPoll, VkEventType
from vk_api.keyboard import VkKeyboard, VkKeyboardColor

class Bot:
    def __init__(self, token):
        self.vk_session = vk_api.VkApi(token=token)
        self.vk = self.vk_session.get_api()
        self.longpoll = VkLongPoll(self.vk_session)

    def send(self, user_id, text, keyboard=None):
        self.vk.messages.send(user_id=user_id, message=text, random_id=0, keyboard=keyboard)

    def main_keyboard(self):
        kb = VkKeyboard(one_time=False)
        kb.add_button('Календарь', color=VkKeyboardColor.PRIMARY)
        kb.add_button('Напомнить', color=VkKeyboardColor.PRIMARY)
        kb.add_line()
        kb.add_button('События', color=VkKeyboardColor.SECONDARY)
        return kb.get_keyboard()

    def run(self):
        print("Bot started")
        for event in self.longpoll.listen():
            if event.type == VkEventType.MESSAGE_NEW and event.to_me:
                self.handle(event.user_id, event.text)

    def handle(self, user_id, text):
        state = db.get_user(user_id)

        if text.lower() == 'календарь':
            db.set_state(user_id, 'calendar')
            self.send(user_id, "Send event data: YYYY-MM-DD HH:MM|Title|Description", self.main_keyboard())

        elif text.lower() == 'напомнить':
            db.set_state(user_id, 'reminder')
            self.send(user_id, "Send reminder: YYYY-MM-DD HH:MM|Text", self.main_keyboard())

        elif text.lower() == 'события':
            events = calendar.get_events()
            if events:
                msg = "Events:\n"
                for e in events:
                    msg += f"- {e['summary']}\n"
            else:
                msg = "No events"
            self.send(user_id, msg, self.main_keyboard())

        elif state == 'calendar' and '|' in text:
            parts = text.split('|')
            if len(parts) >= 2:
                datetime_str, title = parts[0], parts[1]
                desc = parts[2] if len(parts) > 2 else ""

                try:
                    date_part, time_part = datetime_str.split()
                    calendar.add_event(title, desc, date_part, time_part)
                    self.send(user_id, "Event added", self.main_keyboard())
                    db.set_state(user_id, 'menu')
                except:
                    self.send(user_id, "Error: use YYYY-MM-DD HH:MM|Title|Description", self.main_keyboard())

        elif state == 'reminder' and '|' in text:
            # Простое напоминание (без шедулера для простоты)
            self.send(user_id, "Reminder saved (scheduler will be added later)", self.main_keyboard())
            db.set_state(user_id, 'menu')

        else:
            self.send(user_id, "Use buttons", self.main_keyboard())

In [ ]:
# ШАГ 5: Запуск

TOKEN = "your_vk_token_here"  # Замените на ваш токен

# Раскомментируйте для первого запуска Google Calendar
# Нужно будет загрузить credentials.json в папку StudyBot
# calendar.authenticate()

bot = Bot(TOKEN)
bot.run()

from google.colab import files
from google.colab import drive
drive.mount('/content/drive')
path_to_notebook = "/content/drive/MyDrive/Colab Notebooks/Calendar_Testing.ipynb"
files.download(path_to_notebook)